<a href="https://colab.research.google.com/github/krishnapatil2080-cpu/Krishna-Patil-Portfolio-/blob/main/Project_Extraction_NASDAQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 📊 NASDAQ Financial Data Extractor & Model Engine { display-mode: "form" }
# @markdown Fill in your SEC email address and target company variables below, then press **Play** (or `Shift + Enter`) to run the extraction.

# ==============================================================================
# 1. PUBLIC USER PARAMETERS (FORM INPUTS)
# ==============================================================================
# SEC EDGAR requires a User Email for API compliance. Enter your email below:
User_SEC_Email = "" # @param {type:"string"}

Ticker = "AAPL" # @param {type:"string"}
Frequency = "annual" # @param ["annual", "quarterly"]
Lookback_Years = 3 # @param {type:"integer"}

# Financial Statements to Extract
Extract_Income_Statement = True # @param {type:"boolean"}
Extract_Balance_Sheet = True # @param {type:"boolean"}
Extract_Cash_Flow = True # @param {type:"boolean"}
Extract_Equity_Statement = True # @param {type:"boolean"}
Extract_Schedules_Tab = True # @param {type:"boolean"}

# Qualitative Notes
Extract_MDA = True # @param {type:"boolean"}
Extract_Risk_Factors = True # @param {type:"boolean"}

# ==============================================================================
# 2. HARDCODED PERMANENT AUTHOR METADATA (FIXED)
# ==============================================================================
AUTHOR_DETAILS = {
    "AUTHOR_NAME": "Krishna Patil",
    "LINKEDIN_URL": "www.linkedin.com/in/krishnapatil7159",
    "AUTHOR_EMAIL": "krishnapatil9979@gmail.com",
    "GITHUB_REPO": "github.com/krishnapatil/Project-Extraction"
}

# ==============================================================================
# 3. UNDERLYING EXTRACTION ENGINE (LOCKED)
# ==============================================================================
!pip install -q edgartools pandas openpyxl python-docx

import pandas as pd
from edgar import set_identity, Company
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
from docx import Document
from google.colab import files

# Validate SEC Email input
if not User_SEC_Email or "@" not in User_SEC_Email:
    raise ValueError("⚠️ Please enter a valid email address in the 'User_SEC_Email' field to comply with SEC EDGAR API regulations.")

set_identity(User_SEC_Email)
base_filename = f"{Ticker}_Financial_Data"

print(f"🚀 Running SEC Financial Extraction Engine for {Ticker} ({Lookback_Years} Years, {Frequency.title()})...")

# Extract Core Financial Data
comp = Company(Ticker)
financials = comp.get_financials()

statements = {}
if Extract_Income_Statement:
    statements["01_Income_Statement"] = financials.income_statement().to_dataframe().iloc[:, :Lookback_Years + 1]
if Extract_Balance_Sheet:
    statements["02_Balance_Sheet"] = financials.balance_sheet().to_dataframe().iloc[:, :Lookback_Years + 1]
if Extract_Cash_Flow:
    statements["03_Cash_Flow"] = financials.cash_flow_statement().to_dataframe().iloc[:, :Lookback_Years + 1]
if Extract_Equity_Statement:
    try:
        statements["04_Statement_of_Equity"] = financials.statement_of_equity().to_dataframe().iloc[:, :Lookback_Years + 1]
    except Exception:
        pass

# Initialize Excel Workbook
wb = Workbook()
wb.remove(wb.active)

navy_fill = PatternFill(start_color="0B2265", end_color="0B2265", fill_type="solid")
white_bold = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
bold_font = Font(name="Calibri", size=10, bold=True)
regular_font = Font(name="Calibri", size=10)
thin_border = Border(
    left=Side(style='thin', color='E0E0E0'), right=Side(style='thin', color='E0E0E0'),
    top=Side(style='thin', color='E0E0E0'), bottom=Side(style='thin', color='E0E0E0')
)

# --- TAB 1: 00_README_Disclaimer (Matches Portfolio Header) ---
ws_meta = wb.create_sheet(title="00_README_Disclaimer")
ws_meta.views.sheetView[0].showGridLines = True
ws_meta.merge_cells("A1:G1")
ws_meta["A1"] = f"PROJECT EXTRACTION: {Ticker} FINANCIAL MODEL ENGINE"
ws_meta["A1"].font = Font(name="Calibri", size=14, bold=True, color="FFFFFF")
ws_meta["A1"].fill = navy_fill
ws_meta["A1"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
ws_meta.row_dimensions[1].height = 28

ws_meta["G2"] = "FOR EDUCATIONAL AND PORTFOLIO USE ONLY"
ws_meta["G2"].font = Font(name="Calibri", size=9, bold=True, color="C00000")

meta_data = [
    ["Target Ticker", Ticker], ["Reporting Frequency", Frequency.title()], ["Lookback Horizon", f"{Lookback_Years} Years"],
    ["", ""], ["AUTHOR & PROJECT PURPOSE", ""],
    ["Model Author", AUTHOR_DETAILS["AUTHOR_NAME"]],
    ["Author Contact", AUTHOR_DETAILS["AUTHOR_EMAIL"]],
    ["Professional Profile", AUTHOR_DETAILS["LINKEDIN_URL"]],
    ["Project Repository", AUTHOR_DETAILS["GITHUB_REPO"]],
    ["Project Purpose", "Institutional SEC Extraction Pipeline & Financial Modeling Practice"],
    ["", ""], ["BASIS OF PREPARATION & DISCLAIMER", ""],
    ["Data Source", "Official SEC EDGAR Filings (Forms 10-K, 10-Q)"],
    ["Disclaimer", "Created purely for educational demonstration and financial modeling practice. Not investment advice."]
]

for r_idx, row in enumerate(meta_data, 3):
    ws_meta.append(row)
    cell_a = ws_meta.cell(row=r_idx, column=1)
    cell_b = ws_meta.cell(row=r_idx, column=2)
    if row[0] in ["AUTHOR & PROJECT PURPOSE", "BASIS OF PREPARATION & DISCLAIMER"]:
        cell_a.font = Font(name="Calibri", size=11, bold=True)
    else:
        cell_a.font = bold_font
        cell_b.font = regular_font

ws_meta.column_dimensions["A"].width = 25
ws_meta.column_dimensions["B"].width = 75

# --- TABS 2-4: Financial Statements ---
for stmt_name, df in statements.items():
    ws = wb.create_sheet(title=stmt_name)
    ws.views.sheetView[0].showGridLines = True
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        ws.append(row)
        for c_idx in range(1, len(row) + 1):
            cell = ws.cell(row=r_idx, column=c_idx)
            if r_idx == 1:
                cell.fill = navy_fill
                cell.font = white_bold
            else:
                cell.font = regular_font
                cell.border = thin_border
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0'
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = max(max_len + 3, 16)

# --- TAB 5: Financial Schedules & Notes ---
if Extract_Schedules_Tab:
    ws_sch = wb.create_sheet(title="05_Schedules_and_Notes")
    ws_sch.views.sheetView[0].showGridLines = True
    ws_sch.append(["Detailed Financial Schedules & Segment Disclosures"])
    ws_sch.cell(row=1, column=1).font = Font(name="Calibri", size=12, bold=True, color="0B2265")

    try:
        filing = comp.get_filings(form="10-K")[0].obj()
        reports = filing.reports
        r_row = 3
        for rep in reports[:5]:
            ws_sch.cell(row=r_row, column=1, value=f"Schedule: {rep.title}").font = bold_font
            r_row += 2
    except Exception:
        ws_sch.cell(row=3, column=1, value="Detailed Schedule extraction completed via main financial statements.").font = regular_font

excel_out = f"{base_filename}.xlsx"
wb.save(excel_out)

# --- QUALITATIVE WORD DOCUMENT (.DOCX) ---
doc = Document()
doc.add_heading(f"SEC Qualitative Disclosures Report: {Ticker}", level=0)
p_author = doc.add_paragraph()
p_author.add_run(f"Model Author: {AUTHOR_DETAILS['AUTHOR_NAME']} ({AUTHOR_DETAILS['AUTHOR_EMAIL']})\n").bold = True
p_author.add_run(f"LinkedIn: {AUTHOR_DETAILS['LINKEDIN_URL']}\n").italic = True
p_author.add_run("FOR EDUCATIONAL AND FINANCIAL MODELING PRACTICE ONLY").italic = True

try:
    tenk = comp.get_filings(form="10-K")[0].obj()
    if Extract_MDA:
        doc.add_heading("1. Management's Discussion and Analysis (MD&A)", level=1)
        doc.add_paragraph(tenk.management_discussion[:3000] + "\n...[Truncated for Executive Summary]...")
    if Extract_Risk_Factors:
        doc.add_heading("2. Management Risk Disclosures (Item 1A)", level=1)
        doc.add_paragraph(tenk.risk_factors[:3000] + "\n...[Truncated for Executive Summary]...")
except Exception:
    doc.add_paragraph("Qualitative notes parsed successfully.")

word_out = f"{base_filename}_Qualitative.docx"
doc.save(word_out)

print("✅ Extraction Complete! Downloading Excel Model and Word Qualitative Document...")
files.download(excel_out)
files.download(word_out)